In [37]:
from pyspark.sql import SparkSession 
from pyspark.sql import functions as F 
from pyspark.sql import types as T
from pyspark.sql import Window
from pyspark.sql import Row

spark = SparkSession \
    .builder \
    .master("local") \
    .config("spark.driver.memory", "4g") \
    .appName("ex5_google_play_store") \
    .getOrCreate()


In [38]:
google_apps_df = spark.read.csv('s3a://pyspark/data/raw/google_apps/',header=True)

In [40]:
#Define an array of Row objects that map age limits to content ratings.
age_limit_arr = [Row(age_limit = 18,Content_Rating = 'Adults only 18+'),
                 Row(age_limit = 17,Content_Rating = 'Mature 17+'),
                 Row(age_limit =12,Content_Rating = 'Teen'),
                 Row(age_limit=10,Content_Rating = 'Everyone 10+'),
                 Row(age_limit=0,Content_Rating = 'Everyone')]
print(age_limit_arr)

[Row(age_limit=18, Content_Rating='Adults only 18+'), Row(age_limit=17, Content_Rating='Mature 17+'), Row(age_limit=12, Content_Rating='Teen'), Row(age_limit=10, Content_Rating='Everyone 10+'), Row(age_limit=0, Content_Rating='Everyone')]


In [32]:
selected_df = google_apps_df.select(
    F.col('App').alias('application_name'),
    F.col('Category').alias('category'),
    F.col('Rating').alias('rating'),
    F.col('Reviews').cast(T.FloatType()).alias('reviews'),
    F.col('Size').alias('size'),
    F.regexp_replace(F.col("installs"), "[^0-9]", "").cast("int").alias('installs'),
    F.col('Price').cast("Double").alias('price'),
    F.col('Content Rating').alias('content_rating')
).fillna(-1,'Rating')

In [33]:
selected_df.show(100)

+--------------------+-----------------+------+--------+------------------+--------+-----+--------------+
|    application_name|         category|rating| reviews|              size|installs|price|content_rating|
+--------------------+-----------------+------+--------+------------------+--------+-----+--------------+
|Photo Editor & Ca...|   ART_AND_DESIGN|   4.1|   159.0|               19M|   10000|  0.0|      Everyone|
| Coloring book moana|   ART_AND_DESIGN|   3.9|   967.0|               14M|  500000|  0.0|      Everyone|
|U Launcher Lite –...|   ART_AND_DESIGN|   4.7| 87510.0|              8.7M| 5000000|  0.0|      Everyone|
|Sketch - Draw & P...|   ART_AND_DESIGN|   4.5|215644.0|               25M|50000000|  0.0|          Teen|
|Pixel Draw - Numb...|   ART_AND_DESIGN|   4.3|   967.0|              2.8M|  100000|  0.0|      Everyone|
|Paper flowers ins...|   ART_AND_DESIGN|   4.4|   167.0|              5.6M|   50000|  0.0|      Everyone|
|Smoke Effect Phot...|   ART_AND_DESIGN|   3.8